# Section B - Question 1: Robust CIFAR-10 Classification from 500 Training Images

An Inception-style CNN trained from scratch (no pretrained weights, at most 1,000,000 trainable parameters) on `train_small`: 500 CIFAR-10 images, 50 per class, split into 400 training and 100 validation images. Augmentation is applied on the fly to training data only. All hyperparameters are chosen on the validation split only. `clean_test` and `shifted_test` are used only for the final evaluation. Training uses 60 epochs and 3 seeds; results are mean ± sample std.

**Seeds and splits:** dataset split seed 0 (50 images per class, then 40/10 per class train/val); shifted_test seed 1234; hidden-corruption-only test seed 4321; training seeds 0, 1, 2.

**shifted_test pipeline:** random rotation within ±25°, brightness and contrast changes (factors 0.5-1.5), hidden corruption (with probability 0.5 grayscale, otherwise a random permutation of the RGB channels), one black rectangular occluder covering about 25% of the image.

In [ ]:
import os, time, math, random, copy, json
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__, '| torchvision', torchvision.__version__)

# ---------------- reproducibility settings (document these in the report) ----------------
DATA_SEED  = 0          # selects the 50 images per class and the 40/10 train/val split
SHIFT_SEED = 1234       # seeds the shifted_test corruption pipeline
HIDDEN_ONLY_SEED = 4321 # seeds the hidden-corruption-only test (Part E4)
RUN_SEEDS  = [0, 1, 2]  # three training seeds
EPOCHS     = 60         # maximum allowed
BATCH      = 32
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(0)

## Data: train_small (400 train / 100 val), clean_test, shifted_test

In [ ]:
tr_set = datasets.CIFAR10('./data', train=True,  download=True)
te_set = datasets.CIFAR10('./data', train=False, download=True)
def to_t(ds):
    return torch.from_numpy(ds.data).permute(0,3,1,2).contiguous(), torch.tensor(ds.targets)
X_all, y_all = to_t(tr_set)          # uint8, N x 3 x 32 x 32
X_clean, y_clean = to_t(te_set)      # clean_test: original 10,000 test images

# train_small: 50 images per class (500 total), then stratified split 40 train / 10 val per class
rs = np.random.RandomState(DATA_SEED)
tr_idx, va_idx = [], []
for c in range(10):
    idx  = np.where(y_all.numpy() == c)[0]
    pick = rs.choice(idx, 50, replace=False)
    tr_idx += list(pick[:40]); va_idx += list(pick[40:])
tr_idx, va_idx = np.array(tr_idx), np.array(va_idx)
Xtr, ytr = X_all[tr_idx], y_all[tr_idx]
Xva, yva = X_all[va_idx], y_all[va_idx]
print('train:', tuple(Xtr.shape), 'val:', tuple(Xva.shape), '| per-class train:', np.bincount(ytr.numpy()), '| per-class val:', np.bincount(yva.numpy()))
print('DATA_SEED =', DATA_SEED, '| first 5 train idx:', tr_idx[:5], '| first 5 val idx:', va_idx[:5])

In [ ]:
# ---------------- corruption pipeline for shifted_test ----------------
def hidden_corruption(img, rs):
    # Hidden corruption: with probability 0.5 convert to grayscale (3 channels), otherwise randomly permute the RGB channels.
    # img: float tensor 3x32x32 in [0,1].
    if rs.rand() < 0.5:
        g = 0.299*img[0] + 0.587*img[1] + 0.114*img[2]
        return g.unsqueeze(0).repeat(3,1,1)
    return img[torch.from_numpy(rs.permutation(3))]

def corrupt(X, seed, rotate_bc=True, hidden=True, occlude=True):
    # uint8 N x 3 x 32 x 32 -> uint8. Order: rotation, brightness, contrast, hidden, occlusion.
    rs = np.random.RandomState(seed)
    out = torch.empty_like(X)
    for i in range(len(X)):
        img = X[i].float()/255.
        if rotate_bc:
            img = TF.rotate(img, float(rs.uniform(-25, 25)), interpolation=T.InterpolationMode.BILINEAR)
            img = TF.adjust_brightness(img, float(rs.uniform(0.5, 1.5)))
            img = TF.adjust_contrast(img,   float(rs.uniform(0.5, 1.5)))
        if hidden:
            img = hidden_corruption(img, rs)
        if occlude:   # one black rectangle of about 25% of the image area (256 of 1024 pixels)
            r = math.exp(rs.uniform(math.log(0.5), math.log(2.0)))
            h = int(min(32, max(1, round(math.sqrt(256*r)))))
            w = int(min(32, max(1, round(256/h))))
            t = rs.randint(0, 32-h+1); l = rs.randint(0, 32-w+1)
            img[:, t:t+h, l:l+w] = 0.
        out[i] = (img.clamp(0,1)*255).round().to(torch.uint8)
    return out

t0 = time.time()
X_shift  = corrupt(X_clean, SHIFT_SEED)                                                   # shifted_test (all corruptions)
X_hidden = corrupt(X_clean, HIDDEN_ONLY_SEED, rotate_bc=False, hidden=True, occlude=False) # hidden corruption only (Part E4)
print('built test sets in %.1fs' % (time.time()-t0))

fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for j in range(8):
    ax[0,j].imshow(X_clean[j].permute(1,2,0)); ax[0,j].set_title(CLASSES[int(y_clean[j])], fontsize=9); ax[0,j].axis('off')
    ax[1,j].imshow(X_shift[j].permute(1,2,0)); ax[1,j].axis('off')
ax[0,0].text(-0.15, 0.5, 'clean_test', transform=ax[0,0].transAxes, rotation=90, va='center', ha='right')
ax[1,0].text(-0.15, 0.5, 'shifted_test', transform=ax[1,0].transAxes, rotation=90, va='center', ha='right')
fig.suptitle('Clean test images (top) vs the same images after the shift pipeline (bottom)')
plt.tight_layout(); plt.show()

## Model: small Inception CNN (about 150k parameters)

LeNet/AlexNet-style stem (two 3x3 conv-BN-ReLU + max-pool), then two Inception modules (parallel 1x1, 1x1->3x3, 1x1->5x5 and pool->1x1 branches, concatenated), global average pooling, FC -> BatchNorm -> ReLU -> Dropout -> FC. Input normalisation lives inside the model, so every data path (train, val, test, TTA) is normalised identically.

In [ ]:
class ConvBN(nn.Sequential):
    def __init__(self, cin, cout, k):
        super().__init__(nn.Conv2d(cin, cout, k, padding=k//2, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Inception(nn.Module):
    def __init__(self, cin, c1, c3r, c3, c5r, c5, cp):
        super().__init__()
        self.b1 = ConvBN(cin, c1, 1)
        self.b3 = nn.Sequential(ConvBN(cin, c3r, 1), ConvBN(c3r, c3, 3))
        self.b5 = nn.Sequential(ConvBN(cin, c5r, 1), ConvBN(c5r, c5, 5))
        self.bp = nn.Sequential(nn.MaxPool2d(3, 1, 1), ConvBN(cin, cp, 1))
    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], 1)

class Net(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.register_buffer('mean', torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.2470, 0.2435, 0.2616]).view(1,3,1,1))
        self.stem  = nn.Sequential(ConvBN(3,48,3), ConvBN(48,48,3), nn.MaxPool2d(2))   # 16x16
        self.inc1  = Inception(48, 32, 24, 48, 12, 24, 16)                              # -> 120 ch
        self.pool1 = nn.MaxPool2d(2)                                                    # 8x8
        self.inc2  = Inception(120, 64, 48, 96, 16, 32, 32)                             # -> 224 ch
        self.pool2 = nn.MaxPool2d(2)                                                    # 4x4
        self.fc1  = nn.Linear(224, 128)
        self.bn   = nn.BatchNorm1d(128)
        self.drop = nn.Dropout(dropout)       # after the FC layer only (see Part C)
        self.fc2  = nn.Linear(128, 10)
    def forward(self, x):
        x = (x - self.mean) / self.std
        x = self.pool1(self.inc1(self.stem(x)))
        x = self.pool2(self.inc2(x))
        x = x.mean((2, 3))
        x = self.drop(F.relu(self.bn(self.fc1(x))))
        return self.fc2(x)

n_params = sum(p.numel() for p in Net().parameters() if p.requires_grad)
print('trainable parameters:', n_params)
assert n_params <= 1_000_000

## On-the-fly augmentation (training data only)

Augmentations are applied per sample inside Dataset.__getitem__ (geometric, photometric, noise, Cutout) and per batch inside the training loop (Mixup / CutMix). Nothing is pre-generated or saved. Occluders and rotation corners are filled with black (0) because that is what the occlusion in the shifted test looks like.

In [ ]:
class AddNoise(nn.Module):
    def __init__(self, p=0.5, max_sigma=0.06):
        super().__init__(); self.p, self.max_sigma = p, max_sigma
    def forward(self, x):
        if torch.rand(1).item() < self.p:
            x = (x + torch.randn_like(x) * (torch.rand(1).item() * self.max_sigma)).clamp(0, 1)
        return x

def build_transform(cfg):
    t = []
    if cfg.get('geo'):
        t += [T.RandomAffine(degrees=30, scale=(0.9, 1.1)),              # rotation (a bit wider than the +-25 test range) + scale
              T.RandomCrop(32, padding=4, padding_mode='reflect'),       # translation
              T.RandomHorizontalFlip(0.5)]
    if cfg.get('photo'):
        t += [T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.0),
              T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.3),
              AddNoise(0.5, 0.06)]
    if cfg.get('cutout'):
        t += [T.RandomErasing(p=0.7, scale=(0.08, 0.30), ratio=(0.4, 2.5), value=0.0)]
    return T.Compose(t) if t else None

class AugDataset(Dataset):
    def __init__(self, X, y, tf): self.X, self.y, self.tf = X, y, tf
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].float() / 255.
        if self.tf is not None: x = self.tf(x)
        return x, int(self.y[i])

def mix_batch(x, y):
    # with prob 0.7 apply CutMix (50%) or Mixup (50%); returns x, y_a, y_b, lam
    if np.random.rand() > 0.7: return x, y, y, 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    if np.random.rand() < 0.5:                      # CutMix
        lam = np.random.beta(1.0, 1.0); H = W = x.size(2)
        rh, rw = int(H*math.sqrt(1-lam)), int(W*math.sqrt(1-lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy-rh//2, 0), min(cy+rh//2, H); x1, x2 = max(cx-rw//2, 0), min(cx+rw//2, W)
        x = x.clone(); x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1 - (y2-y1)*(x2-x1)/(H*W)
    else:                                           # Mixup
        lam = np.random.beta(0.4, 0.4); x = lam*x + (1-lam)*x[perm]
    return x, y, y[perm], lam

# --- visual check of the training augmentation ---
tf_full = build_transform(dict(geo=1, photo=1, cutout=1)); set_seed(0)
fig, ax = plt.subplots(4, 8, figsize=(16, 8.4))
for j in range(8):
    base = Xtr[j*50]
    ax[0,j].imshow(base.permute(1,2,0)); ax[0,j].set_title('original', fontsize=9); ax[0,j].axis('off')
    for r in range(1, 4):
        ax[r,j].imshow(tf_full(base.float()/255.).permute(1,2,0).clamp(0,1)); ax[r,j].axis('off')
fig.suptitle('Training images (top row) and three random draws of the full on-the-fly augmentation')
plt.tight_layout(); plt.show()

## Training function

SGD (momentum 0.9, Nesterov), batch 32, 3-epoch linear warm-up then cosine learning-rate decay, 60 epochs, weight decay and dropout as arguments, optional label smoothing and Mixup/CutMix. Early stopping is implemented as best-validation checkpoint selection (highest validation accuracy, ties broken by lower validation loss); all 60 epochs are run so learning curves have equal length.

In [ ]:
@torch.no_grad()
def predict_logits(model, X, bs=1000):
    model.eval(); outs = []
    for i in range(0, len(X), bs):
        outs.append(model(X[i:i+bs].to(device).float()/255.).float().cpu())
    return torch.cat(outs)

def acc_of(lg, y): return (lg.argmax(1) == y).float().mean().item() * 100

def train_run(cfg, seed, wd=5e-4, dropout=0.3, epochs=EPOCHS, lr=0.05, warm=3):
    set_seed(seed)
    model = Net(dropout).to(device)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(AugDataset(Xtr, ytr, build_transform(cfg)), batch_size=BATCH, shuffle=True, drop_last=True, generator=g, num_workers=0)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warm if e < warm else 0.5*(1+math.cos(math.pi*(e-warm)/(epochs-warm))))
    ls = cfg.get('ls', 0.0); use_mix = bool(cfg.get('mix'))
    hist = {k: [] for k in ['tr_loss','tr_acc','va_loss','va_acc']}
    best = dict(acc=-1, loss=1e9, ep=-1, state=None)
    for ep in range(epochs):
        model.train(); tl = tc = n = 0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            if use_mix: xb, ya, yb2, lam = mix_batch(xb, yb)
            else: ya, yb2, lam = yb, yb, 1.0
            out = model(xb)
            loss = lam*F.cross_entropy(out, ya, label_smoothing=ls) + (1-lam)*F.cross_entropy(out, yb2, label_smoothing=ls)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tl += loss.item()*len(yb); n += len(yb)
            tc += (out.argmax(1) == (ya if lam >= 0.5 else yb2)).sum().item()
        sched.step()
        lg = predict_logits(model, Xva); vl = F.cross_entropy(lg, yva).item(); va = acc_of(lg, yva)
        hist['tr_loss'].append(tl/n); hist['tr_acc'].append(100*tc/n); hist['va_loss'].append(vl); hist['va_acc'].append(va)
        if va > best['acc'] or (va == best['acc'] and vl < best['loss']):
            best = dict(acc=va, loss=vl, ep=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best['state'])
    tr_clean = acc_of(predict_logits(model, Xtr), ytr)      # accuracy on the 400 clean training images
    return dict(state=best['state'], hist=hist, val_acc=best['acc'], val_loss=best['loss'], best_epoch=best['ep']+1,
                train_acc=tr_clean, dropout=dropout, wd=wd, cfg=cfg)

def load_model(run):
    m = Net(run['dropout']).to(device); m.load_state_dict(run['state']); return m.eval()

## Part D runs: ablation configurations x 3 seeds (training only; no test data touched)

In [ ]:
CFGS = {
 'No augmentation':                   dict(),
 'Geometric only':                    dict(geo=1),
 'Photometric only':                  dict(photo=1),
 'Geometric + photometric':           dict(geo=1, photo=1),
 '+ Cutout / Random Erasing':         dict(geo=1, photo=1, cutout=1),
 '+ CutMix/Mixup + label smoothing':  dict(geo=1, photo=1, cutout=1, mix=1, ls=0.1),
}
results = {}
for name, cfg in CFGS.items():
    results[name] = {}
    for s in RUN_SEEDS:
        t0 = time.time()
        results[name][s] = train_run(cfg, s)
        r = results[name][s]
        print('%-36s seed %d | best val %.1f (epoch %d) | train(clean) %.1f | %.0fs' % (name, s, r['val_acc'], r['best_epoch'], r['train_acc'], time.time()-t0))

## Final model: validation-only hyperparameter selection

Search (seed 0, validation accuracy only; ties broken by validation loss): augmentation level (with or without CutMix/Mixup + label smoothing) x weight decay {5e-4, 3e-3} x dropout {0.3, 0.5}. The winner is then trained with all 3 seeds.

In [ ]:
sweep = {}
for aug_name in ['+ Cutout / Random Erasing', '+ CutMix/Mixup + label smoothing']:
    for wd in [5e-4, 3e-3]:
        for dp in [0.3, 0.5]:
            r = train_run(CFGS[aug_name], 0, wd=wd, dropout=dp)
            sweep[(aug_name, wd, dp)] = r
            print('%-36s wd=%.4f dropout=%.1f -> val %.1f (val loss %.3f)' % (aug_name, wd, dp, r['val_acc'], r['val_loss']))
best_key = max(sweep, key=lambda k: (sweep[k]['val_acc'], -sweep[k]['val_loss']))
print('SELECTED (validation only):', best_key)
FINAL_AUG, FINAL_WD, FINAL_DP = best_key
results['Final model'] = {0: sweep[best_key]}            # same seed + config => identical run, reuse it
for s in RUN_SEEDS[1:]:
    results['Final model'][s] = train_run(CFGS[FINAL_AUG], s, wd=FINAL_WD, dropout=FINAL_DP)
    print('Final model seed', s, '| best val', results['Final model'][s]['val_acc'])

## Test evaluation (done once, after all training and selection)

clean_test, shifted_test and hidden-corruption-only test are evaluated for every stored model. Nothing below feeds back into training or hyperparameter choice.

In [ ]:
test_res = {}
for name in results:
    test_res[name] = {}
    for s in RUN_SEEDS:
        m = load_model(results[name][s])
        lc, ls_, lh = predict_logits(m, X_clean), predict_logits(m, X_shift), predict_logits(m, X_hidden)
        test_res[name][s] = dict(clean=acc_of(lc, y_clean), shifted=acc_of(ls_, y_clean), hidden=acc_of(lh, y_clean),
                                 pred_shift=ls_.argmax(1), prob_shift=F.softmax(ls_, 1))

def ms(vals): return '%.2f +/- %.2f' % (np.mean(vals), np.std(vals, ddof=1))
def collect(name, key):
    return [results[name][s][key] if key in ('val_acc','train_acc') else test_res[name][s][key] for s in RUN_SEEDS]
ORDER = list(results.keys())
print('Std = sample standard deviation (ddof=1) over seeds', RUN_SEEDS)

## Part A - Baseline (no augmentation)

In [ ]:
base = 'No augmentation'
dfA = pd.DataFrame({'Accuracy (%) mean +/- std over 3 seeds': {
    'Training (400 clean imgs)': ms(collect(base,'train_acc')), 'Validation (100 imgs)': ms(collect(base,'val_acc')),
    'Clean test (10,000)': ms(collect(base,'clean')), 'Shifted test (10,000)': ms(collect(base,'shifted'))}})
display(dfA)

def stack(name, key): return np.array([results[name][s]['hist'][key] for s in RUN_SEEDS])
def plot_curves(name, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4)); ep = np.arange(1, EPOCHS+1)
    for (k1, k2, ylab, a) in [('tr_loss','va_loss','Loss',ax[0]), ('tr_acc','va_acc','Accuracy (%)',ax[1])]:
        for key, lab, col in [(k1,'Train','tab:blue'), (k2,'Validation','tab:orange')]:
            arr = stack(name, key); m_, s_ = arr.mean(0), arr.std(0)
            a.plot(ep, m_, color=col, label=lab); a.fill_between(ep, m_-s_, m_+s_, color=col, alpha=0.2)
        a.set_xlabel('Epoch'); a.set_ylabel(ylab); a.set_title(ylab + ' - ' + title); a.grid(alpha=0.3); a.legend()
    plt.tight_layout(); plt.show()
plot_curves(base, 'baseline, no augmentation (mean +/- std, 3 seeds)')

tr, va = np.mean(collect(base,'train_acc')), np.mean(collect(base,'val_acc'))
cl, sh = np.mean(collect(base,'clean')), np.mean(collect(base,'shifted'))
print('Generalisation gap (train - clean test): %.1f points | covariate-shift gap (clean - shifted): %.1f points' % (tr-cl, cl-sh))

In [ ]:
import textwrap
va_curve = stack(base, 'va_loss').mean(0)
best_ep = int(va_curve.argmin()) + 1
overfit_gap, shift_gap = tr - cl, cl - sh
rising = va_curve[-1] > va_curve.min() * 1.05
if overfit_gap > 15 and shift_gap > 5: verdict = 'both overfitting and covariate shift'
elif overfit_gap > 15: verdict = 'mainly overfitting'
elif shift_gap > 5: verdict = 'mainly covariate shift'
else: verdict = 'neither strongly'
diag_text = (f"Training accuracy is {tr:.1f}% versus {va:.1f}% on validation and {cl:.1f}% on clean_test (train-to-clean gap {overfit_gap:.1f} points)"
             + (f"; validation loss is lowest at epoch {best_ep} and then rises to {va_curve[-1]:.2f} while training loss keeps falling, which is memorisation of the 400 images." if rising else ".")
             + f" Accuracy then falls to {sh:.1f}% on shifted_test (clean-to-shifted gap {shift_gap:.1f} points): the network never saw rotated, re-lit, occluded or colour-altered inputs, so the input distribution moves while the labels stay the same (covariate shift)."
             + f" Diagnosis: {verdict}.")
print(textwrap.fill('Part A diagnosis. ' + diag_text, 120))

## Part B - Augmentation design and answers

**Chosen augmentations and why (each one targets a part of the shift):**
- Rotation +-30 degrees, scale 0.9-1.1, reflect-padded random crop (4 px) and horizontal flip: matches the +-25 degree test rotation with a little margin; flip and shift are label-preserving invariances of natural images.
- Colour jitter (brightness 0.5, contrast 0.5, saturation 0.4), random Gaussian blur and additive noise: match the brightness/contrast corruption and add robustness to blur/noise and weak colour cues.
- Cutout / Random Erasing with black fill (p=0.7, 8-30% of the area): directly simulates the ~25% rectangular occlusion.
- CutMix/Mixup + label smoothing 0.1: soft labels and mixed content regularise a tiny dataset, and CutMix patches act as a second kind of occlusion.

**1. Vertical flip: help or hurt?** It mostly hurts here. CIFAR-10 objects have a strong canonical orientation (sky above, ground below; cars, horses, ships, animals stand upright). Upside-down images are almost never present in the test set (rotation is only +-25 degrees), so the model would spend its small capacity on an orientation distribution it never sees, and it blurs cues such as 'sky on top' that help separate airplane/bird/ship from the rest. Horizontal flip is safe; vertical flip is not.

**2. Rotation +-90 degrees when the test only has +-25 degrees.** Wider rotation gives more invariance and acts as a stronger regulariser, but most of the training distribution is then wasted on angles that never appear at test time, the image corners become mostly empty padding, and objects at large angles are harder to recognise, so the effective difficulty rises and the model fits the useful +-25 degree region less precisely. A range slightly above the expected shift (here +-30) is a better trade-off than a much wider one.

**3. Why too much augmentation can reduce accuracy on a small model.** A model with ~150k parameters has limited capacity. If augmentations are too strong (heavy occlusion plus large rotation plus strong colour change plus mixing at once) the training distribution becomes much harder than, and different from, the real test distribution; the model underfits (high training loss), cannot learn fine class-specific features, and in the extreme the label itself becomes ambiguous (e.g. an occluder that hides the whole object). Augmentation strength must be matched to model capacity and to the shift actually expected.

## Part C - Regularisation and training design (justification of placement)

| Technique | Where / setting | Why there |
|---|---|---|
| BatchNorm | after every convolution (and after fc1) | Stabilises training with tiny batches/data, allows a high learning rate, and adds mild noise-like regularisation. It is placed after the linear op and before the ReLU. |
| Dropout (0.3 / 0.5, chosen on validation) | only after the FC hidden layer | The fully connected layer is where co-adaptation and memorisation happen. Convolutional feature maps are spatially correlated, so dropping single activations between early convolutions removes little information and mainly hurts low-level feature learning. |
| Weight decay (5e-4 or 3e-3) | all weights, SGD | Penalises large weights and smooths the function; chosen on validation. |
| Label smoothing 0.1 | loss, in the Mixup/CutMix configuration | Stops over-confident predictions on 400 images and pairs naturally with soft mixed labels. |
| LR schedule | 3-epoch warm-up then cosine decay to ~0 | Warm-up avoids early instability with BN; cosine decay lets the network settle in a flatter minimum at the end. |
| Early stopping | best-validation checkpoint over the 60 epochs | Keeps the weights from the epoch with the best validation accuracy (ties: lower validation loss) instead of the possibly over-fitted last epoch. |
| Global average pooling | replaces large FC layers | Removes most parameters (model has ~150k, limit is 1M) and reduces overfitting. |

## Part D - Ablation table (mean +/- std over 3 seeds)

In [ ]:
dfD = pd.DataFrame({
    'Val Acc':      {n: ms(collect(n,'val_acc')) for n in ORDER},
    'Clean Test':   {n: ms(collect(n,'clean')) for n in ORDER},
    'Shifted Test': {n: ms(collect(n,'shifted')) for n in ORDER}})
dfD.index.name = 'Configuration'
display(dfD)
print('Final model =', FINAL_AUG, '| weight decay', FINAL_WD, '| dropout', FINAL_DP)

fig, ax = plt.subplots(figsize=(11, 4.5)); x = np.arange(len(ORDER)); w = 0.38
for off, key, lab in [(-w/2,'clean','Clean test'), (w/2,'shifted','Shifted test')]:
    ax.bar(x+off, [np.mean(collect(n,key)) for n in ORDER], w, yerr=[np.std(collect(n,key), ddof=1) for n in ORDER], capsize=3, label=lab)
ax.set_xticks(x); ax.set_xticklabels(ORDER, rotation=25, ha='right')
ax.set_ylabel('Accuracy (%)'); ax.set_title('Ablation: clean vs shifted test accuracy (mean +/- std, 3 seeds)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

M = lambda n: np.mean(collect(n,'shifted'))
gains = {'Geometric (vs no augmentation)': M('Geometric only')-M('No augmentation'),
         'Photometric (vs no augmentation)': M('Photometric only')-M('No augmentation'),
         'Cutout / Random Erasing (vs geometric+photometric)': M('+ Cutout / Random Erasing')-M('Geometric + photometric'),
         'CutMix/Mixup + label smoothing (vs +Cutout)': M('+ CutMix/Mixup + label smoothing')-M('+ Cutout / Random Erasing')}
for k, v in gains.items(): print('%-55s %+.2f points on shifted test' % (k, v))
BEST_GAIN = max(gains, key=gains.get)
print('LARGEST SINGLE GAIN:', BEST_GAIN, '(%+.2f points)' % gains[BEST_GAIN])

In [ ]:
MAP = {'Geometric (vs no augmentation)': 'the random rotation of up to +-25 degrees (rotation, shift and scale augmentation)',
       'Photometric (vs no augmentation)': 'the brightness and contrast changes (colour jitter, blur and noise also help against the colour hidden corruption)',
       'Cutout / Random Erasing (vs geometric+photometric)': 'the random rectangular occlusion of about 25% of the image (Random Erasing hides a similar-sized block)',
       'CutMix/Mixup + label smoothing (vs +Cutout)': 'the over-confidence and memorisation caused by having only 400 images (CutMix patches also act as an extra occlusion)'}
g = gains[BEST_GAIN]
pooled_sd = np.mean([np.std(collect(n, 'shifted'), ddof=1) for n in ORDER])
if g <= 0:
    largest_text = 'No single augmentation step improved the mean shifted-test accuracy over the configuration before it.'
else:
    largest_text = (f"{BEST_GAIN.split(' (')[0]} gave the largest gain: {g:+.2f} points of shifted-test accuracy. It matches the shifted-test corruption: {MAP[BEST_GAIN]}."
                    + (f" The gain is similar in size to the average seed-to-seed std ({pooled_sd:.2f}), so it should be read as suggestive." if g < pooled_sd else f" The gain is larger than the average seed-to-seed std ({pooled_sd:.2f})."))
total_gain = M(FN) - M('No augmentation')
print(textwrap.fill(largest_text, 120))
print(f'Total improvement of the final model over the no-augmentation baseline: {total_gain:+.2f} points on shifted_test.')

## Part E - Error analysis (final model)

In [ ]:
FN = 'Final model'
cm = np.zeros((10, 10))
for s in RUN_SEEDS: np.add.at(cm, (y_clean.numpy(), test_res[FN][s]['pred_shift'].numpy()), 1)
cmn = cm / cm.sum(1, keepdims=True)

# per-class accuracy (mean over seeds): clean vs shifted
pc_shift = np.diag(cmn) * 100
cm_clean = np.zeros((10, 10))
for s in RUN_SEEDS:
    pc = predict_logits(load_model(results[FN][s]), X_clean).argmax(1).numpy()
    np.add.at(cm_clean, (y_clean.numpy(), pc), 1)
pc_clean = np.diag(cm_clean / cm_clean.sum(1, keepdims=True)) * 100
display(pd.DataFrame({'Clean acc (%)': pc_clean.round(1), 'Shifted acc (%)': pc_shift.round(1)}, index=CLASSES))

fig, ax = plt.subplots(figsize=(10, 4.2)); x = np.arange(10); w = 0.38
ax.bar(x-w/2, pc_clean, w, label='Clean test'); ax.bar(x+w/2, pc_shift, w, label='Shifted test')
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30); ax.set_ylabel('Accuracy (%)'); ax.set_xlabel('Class')
ax.set_title('Per-class accuracy of the final model (mean over 3 seeds)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10): ax.text(j, i, '%.2f' % cmn[i, j], ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > 0.5 else 'black')
ax.set_xlabel('Predicted class'); ax.set_ylabel('True class'); ax.set_title('Normalised confusion matrix - shifted test, final model (3 seeds pooled)')
plt.colorbar(im, label='Fraction of true class'); plt.tight_layout(); plt.show()

off = [(cmn[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j]
print('Most confused (true -> predicted):')
for v, a, b in sorted(off, reverse=True)[:6]: print('  %-10s -> %-10s %.1f%% of the true class' % (a, b, v*100))

In [ ]:
animals = {'bird', 'cat', 'deer', 'dog', 'frog', 'horse'}; vehicles = {'airplane', 'automobile', 'ship', 'truck'}
REASON = {'animals': 'similar four-legged or compact silhouettes, fur/skin colours and outdoor backgrounds at 32x32; the distinguishing parts (ears, snout, beak) are tiny and are the first things an occluder or blur removes',
          'vehicles': 'similar box-like shapes, wheels and road/sky/water backgrounds; the cues that separate them (cargo bed, wings, hull) are small or cut by the occluder',
          'mixed': 'the dominant background colour (sky/water versus grass/road) takes over once the object is occluded or its colours are altered'}
conf_pairs = sorted(off, reverse=True)[:5]
conf_text = '; '.join(f'{a} -> {b} ({v*100:.1f}%)' for v, a, b in conf_pairs[:3])
order_acc = np.argsort(pc_shift)
print('Best shifted-test classes :', ', '.join(f'{CLASSES[i]} ({pc_shift[i]:.1f}%)' for i in order_acc[::-1][:3]))
print('Worst shifted-test classes:', ', '.join(f'{CLASSES[i]} ({pc_shift[i]:.1f}%)' for i in order_acc[:3]))
print()
for v, a, b in conf_pairs:
    grp = 'animals' if (a in animals and b in animals) else 'vehicles' if (a in vehicles and b in vehicles) else 'mixed'
    print(textwrap.fill(f'{a} -> {b}: {v*100:.1f}% of true {a} images. Why: {REASON[grp]}.', 120))

In [ ]:
# E3: at least 8 misclassified shifted-test images (final model, seed 0) with automatic descriptors
def describe(img_u8):
    x = img_u8.float()/255.
    black = (x.sum(0) == 0).float().mean().item()
    chroma = (x.max(0).values - x.min(0).values).mean().item()
    notes = []
    if black > 0.15: notes.append('large black region (%.0f%% pixels: occluder/rotation corners)' % (black*100))
    if x.mean() < 0.25: notes.append('very dark')
    if x.mean() > 0.65: notes.append('very bright / washed out')
    if chroma < 0.04: notes.append('almost no colour (grayscale-like)')
    if x.std() < 0.12: notes.append('low contrast')
    return '; '.join(notes) if notes else 'object visible but ambiguous / similar-looking class'

pred0, prob0 = test_res[FN][0]['pred_shift'], test_res[FN][0]['prob_shift']
wrong = np.where(pred0.numpy() != y_clean.numpy())[0]
pick = np.random.RandomState(7).choice(wrong, 12, replace=False)
fig, ax = plt.subplots(2, 6, figsize=(15, 6.4)); rows = []
for a, i in zip(ax.ravel(), pick):
    a.imshow(X_shift[i].permute(1, 2, 0)); a.axis('off')
    a.set_title('#%d true: %s\npred: %s (p=%.2f)' % (i, CLASSES[int(y_clean[i])], CLASSES[int(pred0[i])], prob0[i].max()), fontsize=9)
    rows.append((i, CLASSES[int(y_clean[i])], CLASSES[int(pred0[i])], round(prob0[i].max().item(), 2), describe(X_shift[i])))
fig.suptitle('Misclassified shifted-test images (final model, seed 0)'); plt.tight_layout(); plt.show()
pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(rows, columns=['test idx', 'true', 'predicted', 'confidence', 'automatic visual-property note']))

In [ ]:
CAUSE_NOTE = 'Each row above lists the visual property that caused the failure: black occluder/rotation corners hiding the object, extreme brightness or darkness, loss of colour, low contrast, or a similar-looking class.'
print(textwrap.fill(CAUSE_NOTE, 120))

In [ ]:
# E4: hidden corruption only, applied to the clean test set
dfE4 = pd.DataFrame({'Clean test': {n: ms(collect(n,'clean')) for n in ORDER},
                     'Hidden corruption only': {n: ms(collect(n,'hidden')) for n in ORDER},
                     'Full shifted test': {n: ms(collect(n,'shifted')) for n in ORDER}})
dfE4.index.name = 'Configuration'; display(dfE4)
print('Final model: hidden-only acc %.2f vs clean %.2f (drop %.2f) | baseline: hidden-only %.2f vs clean %.2f (drop %.2f)' % (
    np.mean(collect(FN,'hidden')), np.mean(collect(FN,'clean')), np.mean(collect(FN,'clean'))-np.mean(collect(FN,'hidden')),
    np.mean(collect('No augmentation','hidden')), np.mean(collect('No augmentation','clean')), np.mean(collect('No augmentation','clean'))-np.mean(collect('No augmentation','hidden'))))

In [ ]:
fh, ch = np.mean(collect(FN, 'hidden')), np.mean(collect(FN, 'clean'))
bh, bc = np.mean(collect('No augmentation', 'hidden')), np.mean(collect('No augmentation', 'clean'))
drop_f, drop_b = ch - fh, bc - bh
if drop_f <= 0.5 * drop_b: verdict4 = 'generalised well'
elif drop_f < drop_b: verdict4 = 'generalised partially'
else: verdict4 = 'did not generalise'
e4_text = (f"Applying only the hidden corruption to clean_test: final model {fh:.1f}% (drop {drop_f:.1f} points from {ch:.1f}% clean), baseline {bh:.1f}% (drop {drop_b:.1f} points from {bc:.1f}% clean). "
           f"The training augmentation never contains grayscale or channel permutation, only brightness/contrast/saturation jitter, blur, noise, Cutout and CutMix/Mixup, so the final model {verdict4} to the unseen corruption.")
print(textwrap.fill(e4_text, 120))

## Part F (bonus) - Test-time augmentation

In [ ]:
def shift_img(x, dx, dy, p=2):
    xp = F.pad(x, (p, p, p, p), mode='reflect')
    return xp[:, :, p+dy:p+dy+32, p+dx:p+dx+32]
TTA_VIEWS = [(0,0,0), (1,0,0), (0,2,0), (0,-2,0), (0,0,2), (0,0,-2), (1,2,0), (1,-2,0)]   # (hflip, dx, dy)

@torch.no_grad()
def tta_probs(model, X, n):
    model.eval(); total = 0
    for (f, dx, dy) in TTA_VIEWS[:n]:
        ps = []
        for i in range(0, len(X), 1000):
            xb = X[i:i+1000].to(device).float()/255.
            if f: xb = torch.flip(xb, [3])
            ps.append(F.softmax(model(shift_img(xb, dx, dy)), 1).cpu())
        total = total + torch.cat(ps)
    return total / n

rows, curve, tcost = [], {}, {}
for n in [1, 2, 4, 8]:
    accs, times = [], []
    for s in RUN_SEEDS:
        m = load_model(results[FN][s])
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.time(); p = tta_probs(m, X_shift, n)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.time()-t0); accs.append(acc_of(p, y_clean))
    curve[n] = (np.mean(accs), np.std(accs, ddof=1)); tcost[n] = np.mean(times)
    rows.append((n, ms(accs), '%+.2f' % (np.mean(accs)-curve[1][0]), '%.2f s' % np.mean(times), '%dx forward passes' % n))
dfF = pd.DataFrame(rows, columns=['N views', 'Shifted-test acc (%)', 'Change vs N=1 (points)', 'Inference time (10k imgs)', 'Cost'])
display(dfF)
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(list(curve), [v[0] for v in curve.values()], yerr=[v[1] for v in curve.values()], marker='o', capsize=3)
ax.set_xlabel('Number of TTA views N'); ax.set_ylabel('Shifted-test accuracy (%)'); ax.set_title('TTA (flips + 2px shifts) on shifted test'); ax.set_xticks(list(curve)); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
gain8 = curve[8][0] - curve[1][0]; best_n = max(curve, key=lambda k: curve[k][0]); gain_best = curve[best_n][0] - curve[1][0]
tta_text = (f"Averaging softmax over N flipped/shifted views changes shifted-test accuracy by {gain8:+.2f} points at N=8 ({curve[1][0]:.2f}% -> {curve[8][0]:.2f}%; best {gain_best:+.2f} at N={best_n}). "
            f"Inference cost grows linearly with N: {tcost[1]:.2f} s for 10,000 images at N=1 versus {tcost[8]:.2f} s at N=8 ({tcost[8]/max(tcost[1], 1e-9):.1f}x).")
print(textwrap.fill(tta_text, 120))